# Gurobi test: manual cutting planes with Gurobi's own cuts turned off

This notebook checks that a MILP can be solved by Gurobi using only cuts **we write ourselves**, with every built-in cut generator disabled. Two standard cut families are implemented by hand:

1. **Gomory mixed-integer (GMI) cuts**, computed directly from the optimal simplex tableau of the LP relaxation and added over several root rounds.
2. **Extended cover inequalities** for knapsack rows, separated at every branch-and-bound node through a `MIPNODE` callback (`cbCut`).

The enforced solver setup is `Cuts=0` (all Gurobi cut families off), `Presolve=0` (no presolve coefficient tightening, which acts like cutting), and `PreCrush=1` (so user cuts apply to the original model).

**What the tests check**

| # | Test | How |
|---|------|-----|
| 1 | Every manual cut is **valid** | Brute-force enumeration of all integer-feasible points of a small instance |
| 2 | Every manual cut **separates** | It is violated by the fractional point it was derived from |
| 3 | Gurobi's cuts are **really off** | The "Cutting planes" section of the Gurobi log contains only `User` |
| 4 | The root bound **tightens monotonically** and never drops below the MIP optimum | LP bound after each GMI round |
| 5 | The answer is **correct** | Manual-cut objective equals the Gurobi default objective (`MIPGap=0`) |

Test problem: a random multidimensional 0-1 knapsack, $\max\{c^\top x : Ax \le b,\ x \in \{0,1\}^n\}$.

> Note: this uses gurobipy's scalar API (`addVar`, `getRow`) instead of the matrix API (`addMVar`, `getA`), because the matrix API imports `scipy.sparse`, and the scipy install in this environment fails to load.

In [1]:
import itertools
import os
import re
import tempfile

import numpy as np
import gurobipy as gp
from gurobipy import GRB

TOL = 1e-6
ENV = gp.Env(empty=True)
ENV.setParam("OutputFlag", 0)
ENV.start()
print("Gurobi", ".".join(map(str, gp.gurobi.version())))

Gurobi 13.0.3


## Instance generator and model builder

In [2]:
def make_mkp(n, m, seed, tightness=0.5):
    # Random multidimensional 0-1 knapsack: max c'x s.t. Ax <= b, x binary.
    rng = np.random.default_rng(seed)
    A = rng.integers(5, 60, size=(m, n)).astype(float)
    c = (A.sum(axis=0) / m + rng.integers(1, 30, size=n)).round()
    b = np.floor(tightness * A.sum(axis=1))
    return c, A, b


def lin(coef, x):
    # Linear expression sum_j coef_j x_j.
    return gp.LinExpr([float(v) for v in coef if v != 0],
                      [xj for v, xj in zip(coef, x) if v != 0])


def build_model(c, A, b, relax=False, name="mkp"):
    mdl = gp.Model(name, env=ENV)
    vtype = GRB.CONTINUOUS if relax else GRB.BINARY
    x = [mdl.addVar(lb=0.0, ub=1.0, vtype=vtype, name=f"x[{j}]") for j in range(len(c))]
    mdl.setObjective(lin(c, x), GRB.MAXIMIZE)
    for i in range(A.shape[0]):
        mdl.addConstr(lin(A[i], x) <= b[i], name=f"knap[{i}]")
    mdl.update()
    return mdl, x


def dense_A(mdl):
    # Dense constraint matrix of the current model (rows include any added cuts).
    xs = mdl.getVars()
    pos = {v.index: j for j, v in enumerate(xs)}
    cons = mdl.getConstrs()
    A = np.zeros((len(cons), len(xs)))
    for i, con in enumerate(cons):
        row = mdl.getRow(con)
        for k in range(row.size()):
            A[i, pos[row.getVar(k).index]] += row.getCoeff(k)
    return A

## Cut family 1: Gomory mixed-integer cuts from the simplex tableau

gurobipy does not expose tableau rows, so we rebuild them from the optimal basis (`VBasis`, `CBasis`).

**Standard form.** Write the LP as $Ax + Ds = b$ with $l \le x \le u$ and $s \ge 0$, where $D_{ii} = +1$ for a $\le$ row and $-1$ for a $\ge$ row. Let $z = (x, s)$ and $M = [A\ D]$.

**Tableau row.** Shift each nonbasic variable so that it is $0$ at the LP vertex: $y_j = z_j - l_j$ if it sits at its lower bound, and $y_j = u_j - z_j$ if it sits at its upper bound. The tableau row of a basic variable $z_k$ is then
$$z_k + \sum_{j \in N} \bar a_j\, y_j = z_k^*, \qquad \bar a = \pm (B^{-1}N)_{k\cdot}.$$

**GMI cut.** Let $f_0 = \operatorname{frac}(z_k^*) \in (0,1)$ and $f_j = \operatorname{frac}(\bar a_j)$. The cut is
$$\sum_{j\ \text{int},\ f_j \le f_0} \tfrac{f_j}{f_0} y_j \;+\; \sum_{j\ \text{int},\ f_j > f_0} \tfrac{1-f_j}{1-f_0} y_j \;+\; \sum_{j\ \text{cont},\ \bar a_j>0} \tfrac{\bar a_j}{f_0} y_j \;+\; \sum_{j\ \text{cont},\ \bar a_j<0} \tfrac{-\bar a_j}{1-f_0} y_j \;\ge\; 1.$$

Every $y_j$ is $0$ at the LP vertex, so the left-hand side is $0 < 1$ there and the cut is violated by exactly one unit. To get the cut back in $x$-space, substitute $y$ and then $s = D^{-1}(b - Ax)$. Slacks are treated as continuous, which is always valid.

In [3]:
def gmi_cuts_from_basis(mdl, is_int, max_cuts=50, f_min=0.01):
    # Returns a list of (alpha, beta) meaning  alpha @ x >= beta.
    xs, cons = mdl.getVars(), mdl.getConstrs()
    n, m = len(xs), len(cons)
    A = dense_A(mdl)
    b = np.array(mdl.getAttr("RHS", cons))
    d = np.array([1.0 if s == "<" else -1.0 for s in mdl.getAttr("Sense", cons)])
    lb = np.array(mdl.getAttr("LB", xs))
    ub = np.array(mdl.getAttr("UB", xs))
    xval = np.array(mdl.getAttr("X", xs))
    status = np.concatenate([mdl.getAttr("VBasis", xs), mdl.getAttr("CBasis", cons)])

    M = np.hstack([A, np.diag(d)])                       # columns: x, then s
    zval = np.concatenate([xval, d * (b - A @ xval)])
    z_int = np.concatenate([is_int, np.zeros(m, bool)])  # slacks treated as continuous
    z_lb = np.concatenate([lb, np.zeros(m)])
    z_ub = np.concatenate([ub, np.full(m, np.inf)])

    basic = np.flatnonzero(status == 0)
    nonbasic = np.flatnonzero(status != 0)
    assert len(basic) == m, "basis is not square"
    if np.any(status == -3):
        raise RuntimeError("superbasic variables present; solve the LP with simplex")

    Binv_N = np.linalg.solve(M[:, basic], M[:, nonbasic])
    at_upper = status[nonbasic] == -2
    sign = np.where(at_upper, -1.0, 1.0)
    integ = z_int[nonbasic]

    # source rows: fractional basic integer variables, most fractional first
    cand = []
    for r, k in enumerate(basic):
        f0 = zval[k] - np.floor(zval[k])
        if z_int[k] and f_min < f0 < 1 - f_min:
            cand.append((abs(f0 - 0.5), r, f0))
    cand.sort()

    cuts = []
    for _, r, f0 in cand[:max_cuts]:
        a = sign * Binv_N[r]
        fj = a - np.floor(a)
        g = np.zeros_like(a)
        lo, hi = integ & (fj <= f0), integ & (fj > f0)
        pos, neg = ~integ & (a > 0), ~integ & (a < 0)
        g[lo] = fj[lo] / f0
        g[hi] = (1 - fj[hi]) / (1 - f0)
        g[pos] = a[pos] / f0
        g[neg] = -a[neg] / (1 - f0)

        # sum_j g_j y_j >= 1  ->  z-space
        alpha_z, beta = np.zeros(n + m), 1.0
        for gj, j, up in zip(g, nonbasic, at_upper):
            if abs(gj) < 1e-12:
                continue
            if up:
                alpha_z[j] -= gj
                beta -= gj * z_ub[j]
            else:
                alpha_z[j] += gj
                beta += gj * z_lb[j]
        # z-space -> x-space via s = D (b - A x)
        alpha_x = alpha_z[:n] - (alpha_z[n:] * d) @ A
        beta -= (alpha_z[n:] * d) @ b
        alpha_x[np.abs(alpha_x) < 1e-11] = 0.0
        cuts.append((alpha_x, beta))
    return cuts


def gomory_root_rounds(c, A, b, is_int, rounds=10, max_cuts=20):
    # Manual root cutting-plane loop on the LP relaxation.
    # Returns the cuts as (alpha, beta, lp_point_it_was_derived_from, round) and the bound history.
    lp, x = build_model(c, A, b, relax=True, name="root_lp")
    lp.Params.Method = 1      # dual simplex -> vertex solution with a basis
    lp.Params.Presolve = 0
    lp.optimize()
    history, all_cuts = [lp.ObjVal], []
    for rnd in range(rounds):
        xstar = np.array([v.X for v in x])
        cuts = gmi_cuts_from_basis(lp, is_int, max_cuts=max_cuts)
        if not cuts:
            break
        for alpha, beta in cuts:
            all_cuts.append((alpha, beta, xstar, rnd))
            lp.addConstr(lin(alpha, x) >= beta, name=f"gmi_{len(all_cuts)}")
        lp.optimize()
        history.append(lp.ObjVal)
    lp.dispose()
    return all_cuts, history

## Cut family 2: extended cover inequalities (separated in a callback)

For a knapsack row $\sum_j a_j x_j \le \beta$ with $a \ge 0$, a set $C$ is a *cover* if $\sum_{j\in C} a_j > \beta$. Every cover gives the valid inequality $\sum_{j \in C} x_j \le |C|-1$. The *extended* cover $E(C) = C \cup \{j : a_j \ge \max_{i\in C} a_i\}$ gives the stronger cut $\sum_{j\in E(C)} x_j \le |C|-1$.

We separate covers with the standard greedy heuristic: sort items by $(1-x_j^*)/a_j$, add items until the row overflows, then drop items to make the cover minimal. The callback runs at every `MIPNODE` with an optimal relaxation and passes violated cuts to `cbCut`.

In [4]:
def separate_cover(a, rhs, xstar):
    order = np.argsort((1 - xstar) / np.maximum(a, 1e-12))
    C, weight = [], 0.0
    for j in order:
        if a[j] <= 0:
            continue
        C.append(j)
        weight += a[j]
        if weight > rhs + TOL:
            break
    if weight <= rhs + TOL:
        return None
    for j in sorted(C, key=lambda j: xstar[j]):   # make minimal
        if weight - a[j] > rhs + TOL:
            C.remove(j)
            weight -= a[j]
    if np.sum(1 - xstar[C]) >= 1 - TOL:
        return None                               # cover inequality not violated
    amax = a[C].max()
    E = sorted(set(C) | {j for j in range(len(a)) if a[j] >= amax})
    return E, len(C) - 1


def make_cover_callback(A, b, xvars, log):
    def cb(model, where):
        if where != GRB.Callback.MIPNODE:
            return
        if model.cbGet(GRB.Callback.MIPNODE_STATUS) != GRB.OPTIMAL:
            return
        xstar = np.array(model.cbGetNodeRel(xvars))
        for i in range(A.shape[0]):
            res = separate_cover(A[i], b[i], xstar)
            if res is None:
                continue
            E, k = res
            if xstar[E].sum() > k + 1e-4:
                model.cbCut(gp.quicksum(xvars[j] for j in E) <= k)
                log.append((E, k, xstar.copy()))
    return cb

## Solver wrapper

`mode="default"` runs plain Gurobi. `mode="manual"` turns off Gurobi's cuts and presolve, adds the precomputed GMI cuts to the model, and can attach the cover callback.

Gurobi lists the cuts it applied, by family, in the "Cutting planes:" section of its log. We parse that section to check that only `User` cuts were applied.

In [5]:
def parse_cut_section(logfile):
    # {cut family: count} from the 'Cutting planes:' block of a Gurobi log.
    with open(logfile) as fh:
        mt = re.search(r"Cutting planes:\n((?:  .+\n)+)", fh.read())
    if not mt:
        return {}
    out = {}
    for line in mt.group(1).splitlines():
        name, cnt = line.strip().rsplit(":", 1)
        out[name.strip()] = int(cnt)
    return out


MANUAL_PARAMS = {"Cuts": 0, "Presolve": 0, "PreCrush": 1}


def solve(c, A, b, mode, gmi_cuts=(), cover=False, seed=0):
    mdl, x = build_model(c, A, b, name=mode)
    logfile = tempfile.NamedTemporaryFile(suffix=".log", delete=False).name
    mdl.Params.LogToConsole = 0
    mdl.Params.OutputFlag = 1
    mdl.Params.LogFile = logfile
    mdl.Params.MIPGap = 0
    mdl.Params.Seed = seed
    if mode == "manual":
        for k, v in MANUAL_PARAMS.items():
            mdl.setParam(k, v)
        # enforce: the parameters actually took effect
        assert all(mdl.getParamInfo(k)[2] == v for k, v in MANUAL_PARAMS.items())
    for i, (alpha, beta, *_) in enumerate(gmi_cuts):
        mdl.addConstr(lin(alpha, x) >= beta, name=f"gmi_{i}")
    cover_log = []
    mdl.optimize(make_cover_callback(A, b, x, cover_log) if cover else None)
    assert mdl.Status == GRB.OPTIMAL
    res = dict(obj=mdl.ObjVal, bound=mdl.ObjBound, nodes=int(mdl.NodeCount),
               runtime=mdl.Runtime, x=np.array([v.X for v in x]).round(),
               cover_log=cover_log, cut_section=parse_cut_section(logfile))
    mdl.dispose()
    os.remove(logfile)
    return res


def lp_bound(c, A, b, extra_cuts=()):
    lp, x = build_model(c, A, b, relax=True)
    for alpha, beta, *_ in extra_cuts:
        lp.addConstr(lin(alpha, x) >= beta)
    lp.optimize()
    val = lp.ObjVal
    lp.dispose()
    return val

## Test 1 and Test 2: validity and separation on a small instance (exhaustive check)

With $n = 14$ there are $2^{14} = 16{,}384$ binary points. We enumerate all of them, keep the feasible ones, and check every manual cut against every feasible point.

In [6]:
c, A, b = make_mkp(n=14, m=3, seed=1)
is_int = np.ones(len(c), bool)

pts = np.array(list(itertools.product([0, 1], repeat=len(c))), float)
feas = pts[np.all(pts @ A.T <= b + TOL, axis=1)]
z_brute = (feas @ c).max()

gmi, hist = gomory_root_rounds(c, A, b, is_int, rounds=8)
res = solve(c, A, b, "manual", gmi_cuts=gmi, cover=True)

# Test 1: validity -- no integer-feasible point is cut off
for alpha, beta, *_ in gmi:
    assert np.all(feas @ alpha >= beta - 1e-6), "invalid GMI cut"
for E, k, _ in res["cover_log"]:
    assert np.all(feas[:, E].sum(axis=1) <= k + TOL), "invalid cover cut"

# Test 2: separation -- each cut is violated by the point it was generated from
for alpha, beta, xs, _ in gmi:
    assert alpha @ xs < beta - 1e-6, "GMI cut does not separate its LP point"
for E, k, xs in res["cover_log"]:
    assert xs[E].sum() > k + 1e-4, "cover cut does not separate its node relaxation"

assert abs(res["obj"] - z_brute) < 1e-6

print(f"feasible integer points : {len(feas)}")
print(f"brute-force optimum     : {z_brute:.0f}   manual-cut B&C optimum: {res['obj']:.0f}")
print(f"GMI cuts checked        : {len(gmi)}   cover cuts checked: {len(res['cover_log'])}")
print(f"root bound by round     : {np.round(hist, 2)}")
print("Tests 1 and 2 passed")

feasible integer points : 6002
brute-force optimum     : 364   manual-cut B&C optimum: 364
GMI cuts checked        : 22   cover cuts checked: 21
root bound by round     : [379.76 374.85 373.56 373.08 372.84 372.79 372.22 372.14 371.85]
Tests 1 and 2 passed


## Tests 3, 4 and 5: enforcement, bound progress and correctness on a larger instance

We run three configurations on the same 60-variable, 8-row instance:

* **default**: Gurobi with all its built-in cuts.
* **no cuts**: `Cuts=0`, `Presolve=0`, and no manual cuts (the baseline).
* **manual**: same settings as *no cuts*, plus our GMI root rounds and cover callback.

In [7]:
c, A, b = make_mkp(n=60, m=8, seed=7)
is_int = np.ones(len(c), bool)

ref = solve(c, A, b, "default")
base = solve(c, A, b, "manual")
gmi, hist = gomory_root_rounds(c, A, b, is_int, rounds=10)
man = solve(c, A, b, "manual", gmi_cuts=gmi, cover=True)

# Test 3: Gurobi's own cuts are off; only user cuts appear in the log
assert set(ref["cut_section"]) - {"User"}, "sanity: default run should use built-in cuts"
assert base["cut_section"] == {}, f"built-in cuts leaked into baseline: {base['cut_section']}"
assert set(man["cut_section"]) <= {"User"}, f"built-in cuts leaked: {man['cut_section']}"
assert man["cut_section"].get("User", 0) > 0, "no manual cuts were applied"

# Test 4: root bound tightens monotonically and stays a valid upper bound
assert all(h2 <= h1 + 1e-7 for h1, h2 in zip(hist, hist[1:])), "bound not monotone"
assert hist[-1] < hist[0] - 1e-6, "GMI rounds did not tighten the root bound"
assert hist[-1] >= ref["obj"] - 1e-6, "root bound below MIP optimum (invalid cut)"
for alpha, beta, *_ in gmi:
    assert alpha @ ref["x"] >= beta - 1e-6, "GMI cut removes the optimal solution"

# Test 5: same optimum as Gurobi default
assert abs(man["obj"] - ref["obj"]) < 1e-6
assert abs(base["obj"] - ref["obj"]) < 1e-6

z_lp = hist[0]
def gap_closed(bound):
    return 100 * (z_lp - bound) / (z_lp - ref["obj"])

print(f"{'config':<10}{'obj':>9}{'nodes':>9}{'time(s)':>9}   cuts in Gurobi log")
for name, r in [("default", ref), ("no cuts", base), ("manual", man)]:
    print(f"{name:<10}{r['obj']:>9.0f}{r['nodes']:>9}{r['runtime']:>9.2f}   {r['cut_section']}")
print()
print(f"LP bound {z_lp:.2f} -> after {len(hist) - 1} GMI rounds {hist[-1]:.2f}   "
      f"(root gap closed by GMI: {gap_closed(hist[-1]):.1f}%,  {len(gmi)} cuts)")
print(f"distinct cover cuts separated in callback: {len({(tuple(E), k) for E, k, _ in man['cover_log']})}")
print("Tests 3, 4 and 5 passed")

config          obj    nodes  time(s)   cuts in Gurobi log
default        1609     1257     0.07   {'Gomory': 5, 'Cover': 23, 'MIR': 7, 'StrongCG': 5, 'Zero half': 1}
no cuts        1609     2079     0.05   {}
manual         1609     1628     0.58   {'User': 805}

LP bound 1625.16 -> after 10 GMI rounds 1621.17   (root gap closed by GMI: 24.7%,  86 cuts)
distinct cover cuts separated in callback: 2375
Tests 3, 4 and 5 passed


## Robustness sweep

Run the full pipeline on several random instances and seeds. The manual-cut optimum must equal the Gurobi default optimum every time, and Gurobi must never apply its own cuts.

In [8]:
for seed in range(5):
    c, A, b = make_mkp(n=40, m=5, seed=100 + seed)
    is_int = np.ones(len(c), bool)
    ref = solve(c, A, b, "default", seed=seed)
    gmi, hist = gomory_root_rounds(c, A, b, is_int, rounds=6)
    man = solve(c, A, b, "manual", gmi_cuts=gmi, cover=True, seed=seed)
    assert abs(man["obj"] - ref["obj"]) < 1e-6
    assert set(man["cut_section"]) <= {"User"}
    assert all(alpha @ ref["x"] >= beta - 1e-6 for alpha, beta, *_ in gmi)
    print(f"seed {seed}: obj {ref['obj']:.0f}  default nodes {ref['nodes']:>5}  "
          f"manual nodes {man['nodes']:>5}  user cuts {man['cut_section'].get('User', 0)}")
print("\nALL TESTS PASSED")

seed 0: obj 1135  default nodes  1936  manual nodes   859  user cuts 459
seed 1: obj 995  default nodes   190  manual nodes   392  user cuts 253


seed 2: obj 1017  default nodes     1  manual nodes    23  user cuts 12
seed 3: obj 1000  default nodes   366  manual nodes   430  user cuts 327


seed 4: obj 1089  default nodes     1  manual nodes    49  user cuts 22

ALL TESTS PASSED
